# DARTS × GeoAI Arctic — compatibility EDA

**Question.** Can the DARTS RTS training labels (Nitze et al., 3 749 hand-digitised polygons) be
merged with the GeoAI Arctic training set, given that the two come from different imagery?

**Setup.** DARTS ships polygons only (its PlanetScope pixels are not redistributable), so
`fetch_darts_chips.py` rebuilt GeoAI-style 8-band chips over DARTS labels from public data, on
the same 4 m grid, following the Yang et al. (2023) recipe that produced the GeoAI bands:

| GeoAI band | GeoAI source (Yang 2023 + spectra, §2) | DARTS chip source | expected mismatch |
|---|---|---|---|
| red, green, blue | Maxar Vivid 4 m | Sentinel-2 L2A 10 m → 4 m | **sensor + 2.5× resolution** |
| ndvi, ndwi | Sentinel-2 10 m → 4 m | Sentinel-2 10 m → 4 m | none (same pipeline) |
| nir | ~20–30 m source (§2) | Sentinel-2 B8A 20 m → 4 m | tested in §2 |
| relative_elevation | ArcticDEM 2 m − 15 px mean | same | none (recipe matched, §3) |
| shaded_relief | ESR: hillshade + slope, alpha-blended with grey Maxar | same, blended with grey S2 | partly optical (§3) |

Only channels present in both datasets are analysed — that is all 8 GeoAI bands.
Figures are saved next to this notebook as `NN_name.png`.

In [ ]:
import glob, json, os, warnings
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from pycocotools import mask as mask_utils
from scipy import ndimage
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

ROOT = r'C:\Users\moham\Downloads\2026_geoai_arctic_challenge_v1.0'
REL = os.path.join(ROOT, 'competition_release')
OUT = os.path.join(ROOT, 'DARTS_EDA')
CHIPS = os.path.join(OUT, 'chips')
LABEL_DIR = r'C:\Users\moham\Downloads\Darts_dataset\ML_training_labels\retrogressive_thaw_slumps'
BANDS = ['red', 'green', 'blue', 'ndvi', 'relative_elevation', 'shaded_relief', 'nir', 'ndwi']
SHORT = ['red', 'green', 'blue', 'ndvi', 'rel_elev', 'shaded', 'nir', 'ndwi']
OPTICAL, DEM = [0, 1, 2, 3, 6, 7], [4, 5]
GSD = 4.0
C_GEO, C_DARTS = '#1f77b4', '#d62728'
rng = np.random.default_rng(0)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .3})

def save(fig, name):
    fig.savefig(os.path.join(OUT, name), dpi=110, bbox_inches='tight')

## 0. Load both datasets

In [ ]:
# GeoAI: images + instance masks (COCO RLE), joined on basename (see rts-dataset-gotchas)
coco = json.load(open(os.path.join(REL, 'train', 'annotations', 'instances_train.json')))
id2file = {im['id']: os.path.basename(im['file_name']) for im in coco['images']}
anns = {}
for a in coco['annotations']:
    anns.setdefault(id2file[a['image_id']], []).append(a)

geo_img, geo_inst = [], []
for f in sorted(glob.glob(os.path.join(REL, 'train', 'images', '*.npz'))):
    im = np.load(f)['image'].astype(np.float32)
    inst = np.zeros(im.shape[:2], np.int16)
    for k, a in enumerate(anns.get(os.path.basename(f), []), 1):
        inst[mask_utils.decode(a['segmentation']).astype(bool)] = k
    geo_img.append(im); geo_inst.append(inst)

# DARTS chips built by fetch_darts_chips.py
man = pd.read_csv(os.path.join(CHIPS, 'manifest.csv'))
dz = [np.load(os.path.join(CHIPS, c + '.npz')) for c in man.loc[man.status == 'ok', 'chip']]
d_img = [z['image'] for z in dz]
d_inst = [z['inst'] for z in dz]
d_extra = [{k: z[k] for k in z.files if k.startswith('extra_')} for z in dz]
# ndvi/ndwi outside [-1, 1] are impossible: they come from S2 pixels whose reflectance is <= 0
# after the BOA offset, so the denominator is ~0. Mask them (0.02 % of pixels, but they wreck std).
n_bad = 0
for im in d_img:
    for c in (3, 7):
        bad = np.abs(im[..., c]) > 1
        n_bad += bad.sum(); im[..., c][bad] = np.nan
print(f'masked {n_bad} DARTS index pixels outside [-1, 1]')
print(f'GeoAI: {len(geo_img)} chips, {sum(len(v) for v in anns.values())} instances')
print(f'DARTS: {len(d_img)} chips built of {len(man)} attempted')
print(man.status.value_counts().to_string())

## 1. DARTS labels (polygons only, no pixels needed)

The six label iterations are **incremental** (2182 + 743 + 410 + 204 + 141 + 69 = 3749 = the paper's
"3749 features"), so the union is used.

In [ ]:
lab = pd.concat([gpd.read_file(f).assign(iteration=os.path.basename(os.path.dirname(f))[-3:])
                 for f in sorted(glob.glob(os.path.join(LABEL_DIR, '*', 'TrainingLabel_*.gpkg')))],
                ignore_index=True)
lab = gpd.GeoDataFrame(lab, geometry='geometry', crs=4326)
lab['date'] = pd.to_datetime(lab.image_date)
lab['area_m2'] = lab.to_crs(6931).area
cent = lab.geometry.centroid

geo_area_px = np.array([a['area'] for a in coco['annotations']])
d_area_px = lab.area_m2.values / GSD**2

fig, ax = plt.subplots(2, 3, figsize=(17, 9))
lab.iteration.value_counts().sort_index().plot.bar(ax=ax[0, 0], color=C_DARTS); ax[0, 0].set_title('labels per iteration')
lab.region.value_counts().plot.barh(ax=ax[0, 1], color=C_DARTS); ax[0, 1].set_title('labels per region')
pd.crosstab(lab.date.dt.year, lab.date.dt.month).plot.bar(stacked=True, ax=ax[0, 2]); ax[0, 2].set_title('acquisition year × month')
sc = ax[1, 0].scatter(cent.x, cent.y, c=lab.iteration.astype(int), s=4, cmap='viridis'); ax[1, 0].set_title('label centroids (lon/lat)'); plt.colorbar(sc, ax=ax[1, 0], label='iteration')
bins = np.logspace(0, 6, 60)
ax[1, 1].hist(geo_area_px, bins, alpha=.6, density=True, label=f'GeoAI (median {np.median(geo_area_px):.0f} px)', color=C_GEO)
ax[1, 1].hist(d_area_px, bins, alpha=.6, density=True, label=f'DARTS @4 m (median {np.median(d_area_px):.0f} px)', color=C_DARTS)
for t in (300, 2000): ax[1, 1].axvline(t, c='k', ls='--', lw=1)
ax[1, 1].set_xscale('log'); ax[1, 1].set_xlabel('instance area (px on the 4 m grid)'); ax[1, 1].legend(); ax[1, 1].set_title('instance size (scorer bins 300 / 2000 px)')
per_img = lab.groupby('image_id').size()
ax[1, 2].hist(per_img, bins=np.arange(0, 80, 2), color=C_DARTS); ax[1, 2].set_title('labels per PlanetScope image'); ax[1, 2].set_xlabel('count')
plt.tight_layout(); save(fig, '01_darts_labels.png'); plt.show()

def size_bins(a):
    return pd.Series({'small <300': (a < 300).mean(), 'medium 300-2000': ((a >= 300) & (a < 2000)).mean(), 'large ≥2000': (a >= 2000).mean()})
print(pd.DataFrame({'GeoAI': size_bins(geo_area_px), 'DARTS @4 m': size_bins(d_area_px)}).round(3))
print('\nimplied GSD if both populations had the same median slump: '
      f'{np.sqrt(np.median(lab.area_m2) / np.median(geo_area_px)):.2f} m (GeoAI grid is 4 m — DARTS slumps are smaller)')

## 2. Which sensor is behind each GeoAI band? (spatial frequency content)

A band resampled from a coarse source to the 4 m grid loses all detail above that source's
Nyquist frequency. The radially averaged power spectrum therefore shows a *knee* at period
≈ 2 × source GSD / 4 m px. Sentinel-2 10 m → knee at 5 px; 20 m → 10 px.

In [ ]:
def radial_spectrum(x, n=96, nb=48):
    x = np.nan_to_num(x - np.nanmean(x))[:n, :n] * np.outer(np.hanning(n), np.hanning(n))
    P = np.abs(np.fft.fftshift(np.fft.fft2(x)))**2
    yy, xx = np.indices(P.shape); r = np.hypot(yy - n // 2, xx - n // 2) / n
    idx = np.digitize(r, np.linspace(0, .5, nb + 1))
    s = np.array([P[idx == k].mean() for k in range(1, nb + 1)])
    return s / s.sum()

FREQ = (np.arange(48) + .5) * .5 / 48

def mean_spectrum(arrs, n=96):
    S = [radial_spectrum(a, n) for a in arrs if min(a.shape) >= n and np.isfinite(a).mean() > .95]
    return np.mean(S, 0), len(S)

def knee(s):
    L, lf = np.log10(s), np.log10(FREQ)
    d = np.diff(L) / np.diff(lf)
    k = np.argmin(d[4:]) + 4
    return 1 / FREQ[k]

spec_geo = {b: mean_spectrum([im[..., i] for im in geo_img])[0] for i, b in enumerate(SHORT)}
spec_d = {b: mean_spectrum([im[..., i] for im in d_img])[0] for i, b in enumerate(SHORT)}
spec_d['nir_B08_10m'] = mean_spectrum([e['extra_nir_b08'] for e in d_extra])[0]

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for b in SHORT:
    ax[0].loglog(FREQ, spec_geo[b], label=f'{b} (knee {knee(spec_geo[b]):.1f} px)')
ax[0].set_title('GeoAI: power spectrum per band'); ax[0].set_xlabel('cycles / px'); ax[0].legend(fontsize=8)
for b, c in [('nir', 'k'), ('ndvi', 'g'), ('red', 'r')]:
    ax[1].loglog(FREQ, spec_geo[b], c=c, label=f'GeoAI {b}')
ax[1].loglog(FREQ, spec_d['nir'], 'k--', label='DARTS S2 B8A 20 m')
ax[1].loglog(FREQ, spec_d['nir_B08_10m'], 'k:', label='DARTS S2 B08 10 m')
ax[1].loglog(FREQ, spec_d['ndvi'], 'g--', label='DARTS S2 ndvi 10 m')
ax[1].loglog(FREQ, spec_d['red'], 'r--', label='DARTS S2 red 10 m')
ax[1].set_title('source check: GeoAI (solid) vs S2 rebuilds (dashed)'); ax[1].legend(fontsize=8); ax[1].set_xlabel('cycles / px')
# distance between spectral shapes: log-space RMS over the informative range
def spec_dist(a, b, lo=2, hi=40):
    return np.sqrt(np.mean((np.log10(a[lo:hi]) - np.log10(b[lo:hi]))**2))
cands = {'nir vs S2 B8A (20 m)': spec_dist(spec_geo['nir'], spec_d['nir']),
         'nir vs S2 B08 (10 m)': spec_dist(spec_geo['nir'], spec_d['nir_B08_10m']),
         'ndvi vs S2 ndvi (10 m)': spec_dist(spec_geo['ndvi'], spec_d['ndvi']),
         'ndwi vs S2 ndwi (10 m)': spec_dist(spec_geo['ndwi'], spec_d['ndwi']),
         'red vs S2 red (10 m)': spec_dist(spec_geo['red'], spec_d['red'])}
ax[2].barh(list(cands), list(cands.values()), color='grey'); ax[2].set_title('spectral-shape distance (log RMS, lower = same source)')
plt.tight_layout(); save(fig, '02_band_source_spectra.png'); plt.show()
print(pd.Series({b: knee(spec_geo[b]) for b in SHORT}, name='GeoAI knee period (px)').round(1).to_string())
print(); print(pd.Series(cands).round(3).to_string())

## 3. Rebuilding the DEM bands with the GeoAI recipe

**relative_elevation** (Yang 2023: DEM − 15 px mean filter). Candidates stored by the fetcher:
derive at 2 m with a 15 px kernel then average to 4 m (paper order), or average first and filter at 4 m.

**shaded_relief** = enhanced shaded relief: weighted multi-directional hillshade + slope (equal
weights), then alpha-blended with greyscale optical imagery. Altitude, slope scaling, blend
alpha and any smoothing are not published, so they are fitted here against the GeoAI band:
shape of the z-scored distribution, power-spectrum shape, and correlation with grey RGB.

In [ ]:
def pool(arrs, ch=None, n=300_000, mask=None):
    v = np.concatenate([(a if ch is None else a[..., ch]).ravel() for a in arrs])
    v = v[np.isfinite(v)]
    return rng.choice(v, min(n, len(v)), replace=False)

def zs(v):
    return (v - np.median(v)) / (np.subtract(*np.percentile(v, [75, 25])) + 1e-9)

def rts_auc(imgs, insts, ch_fn, n=400_000):
    x, y = [], []
    for im, m in zip(imgs, insts):
        v = ch_fn(im)
        ok = np.isfinite(v)
        x.append(v[ok]); y.append(m[ok] > 0)
    x, y = np.concatenate(x), np.concatenate(y)
    i = rng.choice(len(x), min(n, len(x)), replace=False)
    return roc_auc_score(y[i], x[i])

def nan_mean_filter(z, k):
    ok = np.isfinite(z)
    num = ndimage.uniform_filter(np.where(ok, z, 0.), k, mode='nearest')
    den = ndimage.uniform_filter(ok.astype(float), k, mode='nearest')
    return np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)

# --- relative elevation: window size in metres is what sets the amplitude, so scan it
geo_re = pool(geo_img, 4)
geo_iqr = np.subtract(*np.percentile(geo_re, [75, 25]))
re_cands = {'2 m, k15 → 4 m (paper order, 30 m)': [im[..., 4] for im in d_img],
            '4 m, k8 (32 m)': [e['extra_re_4m_k8'] for e in d_extra],
            '4 m, k15 (60 m)': [e['extra_re_4m_k15'] for e in d_extra]}
for k in (25, 35, 51):
    re_cands[f'4 m, k{k} ({4 * k} m)'] = [e['extra_dem'] - nan_mean_filter(e['extra_dem'], k) for e in d_extra]
rows = []
for name, arrs in re_cands.items():
    v = pool(arrs)
    rows.append({'candidate': name, 'W(z-scored)': wasserstein_distance(zs(geo_re), zs(v)),
                 'log IQR ratio': abs(np.log(np.subtract(*np.percentile(v, [75, 25])) / geo_iqr)),
                 'IQR': np.subtract(*np.percentile(v, [75, 25])), 'p1': np.percentile(v, 1), 'p99': np.percentile(v, 99),
                 'RTS AUC': rts_auc([a[..., None] for a in arrs], d_inst, lambda a: a[..., 0]),
                 'spectral dist': spec_dist(spec_geo['rel_elev'], mean_spectrum(arrs)[0])})
rows.append({'candidate': 'GeoAI (target)', 'W(z-scored)': 0, 'IQR': np.subtract(*np.percentile(geo_re, [75, 25])),
             'p1': np.percentile(geo_re, 1), 'p99': np.percentile(geo_re, 99),
             'RTS AUC': rts_auc(geo_img, geo_inst, lambda a: a[..., 4]), 'spectral dist': 0})
re_tab = pd.DataFrame(rows).set_index('candidate')
cand = re_tab.drop('GeoAI (target)')
best_re = (cand['log IQR ratio'].rank() + cand['W(z-scored)'].rank() + cand['spectral dist'].rank()).idxmin()
print(re_tab.drop(columns='log IQR ratio').round(3).to_string())
print('\nchosen relative-elevation recipe:', best_re)
for im, r in zip(d_img, re_cands[best_re]):
    im[..., 4] = r

In [ ]:
# --- enhanced shaded relief fit
geo_sh = pool(geo_img, 5)
gs =np.concatenate([im[..., 5].ravel() for im in geo_img[::3]]); gg = np.concatenate([im[..., :3].mean(-1).ravel() for im in geo_img[::3]])
ok = np.isfinite(gs) & np.isfinite(gg); geo_grey_corr = np.corrcoef(gs[ok], gg[ok])[0, 1]
print(f'GeoAI corr(shaded_relief, grey RGB) = {geo_grey_corr:.3f}')

def grey255(im):
    g = im[..., :3].mean(-1)
    lo, hi = np.nanpercentile(g, [2, 98])
    return 255 * np.clip((g - lo) / (hi - lo + 1e-6), 0, 1)

def esr(im, ex, alt, w_slope, alpha, sigma):
    hs = ex[f'extra_hs_alt{alt}']
    sl = 255 * (1 - np.clip(ex['extra_slope_deg'] / 45, 0, 1))
    base = (1 - w_slope) * hs + w_slope * sl
    out = (1 - alpha) * base + alpha * grey255(im)
    return ndimage.gaussian_filter(np.nan_to_num(out, nan=np.nanmean(out)), sigma) if sigma else out

sub = list(range(0, len(d_img), 2))
res = []
for alt in (30, 45, 60):
    for w_slope in (0, .5):
        for alpha in (0, .25, .5):
            for sigma in (0, 1, 2, 3):
                arrs = [esr(d_img[i], d_extra[i], alt, w_slope, alpha, sigma) for i in sub]
                v = pool(arrs, n=100_000)
                sp = mean_spectrum(arrs)[0]
                ea = np.concatenate([a.ravel() for a in arrs])
                eg = np.concatenate([d_img[i][..., :3].mean(-1).ravel() for i in sub])
                m = np.isfinite(ea) & np.isfinite(eg)  # S2 cloud/DEM voids are NaN
                cg = np.corrcoef(ea[m], eg[m])[0, 1]
                res.append(dict(alt=alt, w_slope=w_slope, alpha=alpha, sigma=sigma,
                                W_z=wasserstein_distance(zs(geo_sh), zs(v)),
                                spec=spec_dist(spec_geo['shaded'], sp), corr_grey=cg))
esr_tab = pd.DataFrame(res)
for c in ('W_z', 'spec'):
    esr_tab[c + '_rank'] = esr_tab[c].rank()
esr_tab['corr_err'] = (esr_tab.corr_grey - geo_grey_corr).abs()
esr_tab['score'] = esr_tab.W_z_rank + esr_tab.spec_rank + esr_tab.corr_err.rank()
best = esr_tab.sort_values('score').iloc[0]
print(esr_tab.sort_values('score').head(8).round(3).to_string(index=False))
BEST_ESR = dict(alt=int(best.alt), w_slope=best.w_slope, alpha=best.alpha, sigma=int(best.sigma))
print('\nchosen ESR recipe:', BEST_ESR)

# rebuild the DARTS shaded band with the fitted recipe (affine-matched to the GeoAI scale)
raw = [esr(im, ex, **BEST_ESR) for im, ex in zip(d_img, d_extra)]
rv = pool(raw)
a = np.std(geo_sh) / np.std(rv); b = np.median(geo_sh) - a * np.median(rv)
for im, r in zip(d_img, raw):
    im[..., 5] = a * r + b

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].hist(zs(geo_re), np.linspace(-6, 6, 120), density=True, alpha=.6, label='GeoAI', color=C_GEO)
ax[0].hist(zs(pool([im[..., 4] for im in d_img])), np.linspace(-6, 6, 120), density=True, alpha=.6, label='DARTS rebuild', color=C_DARTS)
ax[0].set_title('relative elevation (robust z)'); ax[0].legend()
ax[1].hist(geo_sh, 80, density=True, alpha=.6, label='GeoAI', color=C_GEO)
ax[1].hist(pool([im[..., 5] for im in d_img]), 80, density=True, alpha=.6, label='DARTS ESR (fitted)', color=C_DARTS)
ax[1].set_title('shaded relief'); ax[1].legend()
ax[2].loglog(FREQ, spec_geo['shaded'], c=C_GEO, label='GeoAI shaded')
ax[2].loglog(FREQ, mean_spectrum([im[..., 5] for im in d_img])[0], c=C_DARTS, ls='--', label='DARTS ESR')
ax[2].loglog(FREQ, spec_geo['rel_elev'], c=C_GEO, alpha=.5, label='GeoAI rel elev')
ax[2].loglog(FREQ, mean_spectrum([im[..., 4] for im in d_img])[0], c=C_DARTS, alpha=.5, ls='--', label='DARTS rel elev')
ax[2].legend(fontsize=8); ax[2].set_title('spectra'); ax[2].set_xlabel('cycles / px')
plt.tight_layout(); save(fig, '03_dem_recipe_match.png'); plt.show()

## 4. Per-channel EDA on the shared channels (raw values, train-ready 4 m view)

In [ ]:
def stats(imgs):
    rows = {}
    for i, b in enumerate(SHORT):
        allv = np.concatenate([im[..., i].ravel() for im in imgs])
        v = allv[np.isfinite(allv)]
        rows[b] = {'median': np.median(v), 'p1': np.percentile(v, 1), 'p99': np.percentile(v, 99),
                   'std': v.std(), 'nan %': 100 * (1 - len(v) / len(allv))}
    return pd.DataFrame(rows).T

st = pd.concat({'GeoAI': stats(geo_img), 'DARTS (S2/ArcticDEM)': stats(d_img)}, axis=1)
print(st.round(3).to_string())

fig, ax = plt.subplots(2, 4, figsize=(19, 8))
for i, b in enumerate(SHORT):
    g, d = pool(geo_img, i), pool(d_img, i)
    lo, hi = np.percentile(np.r_[g, d], [.5, 99.5])
    bins = np.linspace(lo, hi, 90)
    a_ = ax.flat[i]
    a_.hist(g, bins, density=True, alpha=.55, color=C_GEO, label='GeoAI')
    a_.hist(d, bins, density=True, alpha=.55, color=C_DARTS, label='DARTS')
    a_.set_title(b); a_.legend(fontsize=8)
fig.suptitle('raw per-channel distributions'); plt.tight_layout(); save(fig, '04_raw_histograms.png'); plt.show()

In [ ]:
def corr_matrix(imgs, n=300_000):
    k = n // len(imgs) + 1  # sample per chip instead of concatenating every pixel
    X = np.concatenate([im.reshape(-1, 8)[rng.choice(im.shape[0] * im.shape[1], min(k, im.shape[0] * im.shape[1]), replace=False)]
                        for im in imgs])
    X = X[np.isfinite(X).all(1)]
    return np.corrcoef(X.T)

cg, cd = corr_matrix(geo_img), corr_matrix(d_img)
fig, ax = plt.subplots(1, 3, figsize=(19, 5.5))
for a_, M, t in zip(ax, [cg, cd, cd - cg], ['GeoAI', 'DARTS', 'DARTS − GeoAI']):
    h = a_.imshow(M, vmin=-1, vmax=1, cmap='RdBu_r')
    a_.set_xticks(range(8), SHORT, rotation=60); a_.set_yticks(range(8), SHORT); a_.set_title(t); a_.grid(False)
    for (r, c), v in np.ndenumerate(M): a_.text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(h, ax=ax); save(fig, '05_band_correlations.png'); plt.show()

In [ ]:
auc = pd.DataFrame({'GeoAI': [rts_auc(geo_img, geo_inst, lambda a, i=i: a[..., i]) for i in range(8)],
                    'DARTS': [rts_auc(d_img, d_inst, lambda a, i=i: a[..., i]) for i in range(8)]}, index=SHORT)
auc['GeoAI dir'] = np.where(auc.GeoAI > .5, 'higher in RTS', 'lower in RTS')
auc['DARTS dir'] = np.where(auc.DARTS > .5, 'higher in RTS', 'lower in RTS')
auc['agree'] = auc['GeoAI dir'] == auc['DARTS dir']
print(auc.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(8)
ax.bar(x - .2, auc.GeoAI - .5, .4, color=C_GEO, label='GeoAI'); ax.bar(x + .2, auc.DARTS - .5, .4, color=C_DARTS, label='DARTS')
ax.axhline(0, c='k', lw=.8); ax.set_xticks(x, SHORT); ax.set_ylabel('RTS-vs-background AUC − 0.5'); ax.legend()
ax.set_title('RTS signature per channel (same sign = same cue)'); save(fig, '06_rts_signature.png'); plt.show()

## 5. Resolution mismatch (only red / green / blue are sensor-mismatched in resolution)

* **Fair-stats view**: GeoAI RGB area-averaged to 10 m — radiometric gaps that survive are sensor/processing, not sharpness.
* **Sharpness**: Laplacian variance of RGB luminance per chip and RGB spectra for GeoAI native, GeoAI 4→10→4 m (simulated S2), and DARTS (true S2).

In [ ]:
def down_up(a, f=2.5):
    # area-average to 10 m then bilinear back to the 4 m grid
    small = ndimage.zoom(ndimage.uniform_filter(np.nan_to_num(a, nan=np.nanmean(a)), size=3), 1 / f, order=1)
    return ndimage.zoom(small, np.array(a.shape) / np.array(small.shape), order=1)

def lum(im):
    return im[..., :3].mean(-1)

def lapvar(x):
    x = (x - np.nanmean(x)) / (np.nanstd(x) + 1e-9)
    return np.nanvar(ndimage.laplace(np.nan_to_num(x)))

geo_lum = [lum(im) for im in geo_img]
geo_lum_du = [down_up(l) for l in geo_lum]
d_lum = [lum(im) for im in d_img]
lv = pd.DataFrame({'GeoAI native': pd.Series([lapvar(l) for l in geo_lum]),
                   'GeoAI 4→10→4 m': pd.Series([lapvar(l) for l in geo_lum_du]),
                   'DARTS (S2)': pd.Series([lapvar(l) for l in d_lum])})
print('Laplacian variance of standardised luminance (sharpness):'); print(lv.describe().round(4).T)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].loglog(FREQ, mean_spectrum(geo_lum)[0], c=C_GEO, label='GeoAI native (Maxar)')
ax[0].loglog(FREQ, mean_spectrum(geo_lum_du)[0], c=C_GEO, ls=':', label='GeoAI 4→10→4 m')
ax[0].loglog(FREQ, mean_spectrum(d_lum)[0], c=C_DARTS, ls='--', label='DARTS (S2 10 m → 4 m)')
ax[0].set_title('RGB luminance spectrum'); ax[0].legend(); ax[0].set_xlabel('cycles / px')
for c, col in zip(lv.columns, [C_GEO, 'steelblue', C_DARTS]):
    ax[1].hist(np.log10(lv[c]), 40, alpha=.5, density=True, label=c, color=col)
ax[1].set_xlabel('log10 Laplacian variance'); ax[1].legend(); ax[1].set_title('per-chip sharpness')
plt.tight_layout(); save(fig, '07_resolution_gap.png'); plt.show()

sd = spec_dist(mean_spectrum(d_lum)[0], mean_spectrum(geo_lum_du)[0]); sn = spec_dist(mean_spectrum(d_lum)[0], mean_spectrum(geo_lum)[0])
print(f'spectral distance DARTS vs GeoAI-degraded-to-10 m: {sd:.3f}   vs GeoAI native: {sn:.3f}')
del geo_lum, geo_lum_du, d_lum  # ~0.5 GB, not needed again

# small-object loss at 10 m
for t in (100, 400, 900):
    print(f'DARTS labels < {t:4d} m² ({t // 100} S2 px): {(lab.area_m2 < t).mean():.1%}   '
          f'GeoAI instances < {t:4d} m²: {(geo_area_px * GSD**2 < t).mean():.1%}')

## 6. Does normalisation close the gap?

Schemes applied to both datasets, per channel:
1. **raw**
2. **dataset z**: each dataset standardised with its own median / IQR
3. **chip z**: per-chip mean / std
4. **chip p2–98**: per-chip percentile stretch to [0, 1]
5. **hist-match**: DARTS mapped onto the pooled GeoAI quantiles (whole dataset; DEM channels get dataset-z)

Metrics: Wasserstein distance in GeoAI-σ units, and a **domain classifier** (can a model tell
which dataset a chip came from?) — AUC 0.5 = indistinguishable.

In [ ]:
def ds_stats(imgs):
    X = np.concatenate([im.reshape(-1, 8) for im in imgs[::2]])
    med = np.nanmedian(X, 0); iqr = np.nanpercentile(X, 75, 0) - np.nanpercentile(X, 25, 0)
    return med, iqr + 1e-9

GEO_MED, GEO_IQR = ds_stats(geo_img)
D_MED, D_IQR = ds_stats(d_img)
QS = np.linspace(0, 100, 1001)
GEO_Q = np.nanpercentile(np.concatenate([im.reshape(-1, 8) for im in geo_img[::2]]), QS, axis=0)
D_Q = np.nanpercentile(np.concatenate([im.reshape(-1, 8) for im in d_img[::2]]), QS, axis=0)

def normalise(im, scheme, src):
    im = im.astype(np.float32)
    if scheme == 'raw':
        return im
    if scheme == 'dataset z':
        med, iqr = (GEO_MED, GEO_IQR) if src == 'geo' else (D_MED, D_IQR)
        return (im - med) / iqr
    if scheme == 'chip z':
        return (im - np.nanmean(im, (0, 1))) / (np.nanstd(im, (0, 1)) + 1e-6)
    if scheme == 'chip p2-98':
        lo, hi = np.nanpercentile(im, 2, (0, 1)), np.nanpercentile(im, 98, (0, 1))
        return np.clip((im - lo) / (hi - lo + 1e-6), 0, 1)
    if scheme == 'hist-match':
        if src == 'geo':
            return (im - GEO_MED) / GEO_IQR
        out = (im - D_MED) / D_IQR
        for c in OPTICAL:
            out[..., c] = (np.interp(im[..., c], D_Q[:, c], GEO_Q[:, c]) - GEO_MED[c]) / GEO_IQR[c]
        return out

SCHEMES = ['raw', 'dataset z', 'chip z', 'chip p2-98', 'hist-match']

def chip_features(im, chans, texture):
    f = []
    for c in chans:
        v = im[..., c]; v = v[np.isfinite(v)]
        f += [v.mean(), v.std(), *np.percentile(v, [10, 50, 90])] if len(v) else [0] * 5
    if len(chans) > 1:
        X = im[..., chans].reshape(-1, len(chans)); X = X[np.isfinite(X).all(1)]
        C = np.corrcoef(X.T) if len(X) > 10 else np.zeros((len(chans), len(chans)))
        f += list(np.nan_to_num(C[np.triu_indices(len(chans), 1)]))
    if texture:
        for c in chans:
            f.append(np.log10(lapvar(im[..., c]) + 1e-9))
    return f

GROUPS = {'RGB': [0, 1, 2], 'nir+indices': [3, 6, 7], 'DEM (rel, shaded)': [4, 5], 'all 8': list(range(8))}

def scheme_pass(s, px_geo=300, px_darts=700):
    # One streaming pass per scheme: each chip is normalised, reduced to per-chip features and a
    # fixed pixel sample (+ RTS label), then dropped. A full normalised copy of GeoAI is 1-2 GB
    # and got the notebook killed for memory twice.
    out = {}
    for src, imgs, insts, k in (('geo', geo_img, geo_inst, px_geo), ('darts', d_img, d_inst, px_darts)):
        F = {g: [] for g in GROUPS}
        pix, lab_ = [], []
        for im, m in zip(imgs, insts):
            x = normalise(im, s, src)
            for g, ch in GROUPS.items():
                F[g].append(chip_features(x, ch, True))  # texture features are the last len(ch)
            sel = rng.choice(x.shape[0] * x.shape[1], min(k, x.shape[0] * x.shape[1]), replace=False)
            pix.append(x.reshape(-1, 8)[sel]); lab_.append(m.ravel()[sel] > 0)
        out[src] = {'feat': {g: np.nan_to_num(np.array(v, float)) for g, v in F.items()},
                    'pix': np.concatenate(pix), 'rts': np.concatenate(lab_)}
    return out

PASS = {}
for s in SCHEMES:
    PASS[s] = scheme_pass(s)

W = pd.DataFrame(index=SCHEMES, columns=SHORT, dtype=float)
KS = W.copy()
for s in SCHEMES:
    for i, b in enumerate(SHORT):
        g, d = PASS[s]['geo']['pix'][:, i], PASS[s]['darts']['pix'][:, i]
        g, d = g[np.isfinite(g)], d[np.isfinite(d)]
        W.loc[s, b] = wasserstein_distance(g, d) / (g.std() + 1e-9)
        KS.loc[s, b] = ks_2samp(g, d).statistic
print('Wasserstein distance (GeoAI σ units)'); print(W.round(3).to_string())
print('\nKS statistic'); print(KS.round(3).to_string())

def domain_auc(s, group, texture):
    ch = GROUPS[group]
    Xg, Xd = PASS[s]['geo']['feat'][group], PASS[s]['darts']['feat'][group]
    if not texture:
        Xg, Xd = Xg[:, :-len(ch)], Xd[:, :-len(ch)]
    X = np.r_[Xg, Xd]
    y = np.r_[np.zeros(len(Xg)), np.ones(len(Xd))]
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    out = {}
    for name, clf in [('logreg', make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=.5))),
                      ('gbm', HistGradientBoostingClassifier(max_iter=200, learning_rate=.05))]:
        p = cross_val_predict(clf, X, y, cv=cv, method='predict_proba')[:, 1]
        out[name] = roc_auc_score(y, p)
    return max(out.values())

rows = []
for s in SCHEMES:
    for gname in GROUPS:
        for tex in (False, True):
            rows.append({'scheme': s, 'group': gname, 'texture': tex, 'domain AUC': domain_auc(s, gname, tex)})
dom = pd.DataFrame(rows)
dom_tab = dom.pivot_table(index=['group', 'texture'], columns='scheme', values='domain AUC')[SCHEMES]
print(dom_tab.round(3).to_string())

In [ ]:
# what gives the source away? single-feature AUC (0.5 = useless, 0 or 1 = a perfect giveaway)
def feature_names(chans, texture):
    n = [f'{SHORT[c]} {s}' for c in chans for s in ('mean', 'std', 'p10', 'p50', 'p90')]
    n += [f'corr {SHORT[a]}–{SHORT[b]}' for a, b in zip(*np.triu_indices(len(chans), 1)) for a, b in [(chans[a], chans[b])]]
    return n + ([f'{SHORT[c]} sharpness' for c in chans] if texture else [])

give = {}
for s in ['chip z', 'hist-match']:
    Xg, Xd = PASS[s]['geo']['feat']['all 8'], PASS[s]['darts']['feat']['all 8']
    X, y = np.r_[Xg, Xd], np.r_[np.zeros(len(Xg)), np.ones(len(Xd))]
    a = pd.Series([roc_auc_score(y, X[:, j]) for j in range(X.shape[1])], index=feature_names(list(range(8)), True))
    give[s] = a.iloc[np.argsort(-np.abs(a.values - .5))]
for s, a in give.items():
    print(f'\n{s}: 12 most revealing features (AUC > .5 means higher in DARTS)')
    print(a.head(12).round(3).to_string())

In [ ]:
# does normalisation keep the RTS cue direction?
def sample_auc(p, i):
    v, y = p['pix'][:, i], p['rts']
    ok = np.isfinite(v)
    return roc_auc_score(y[ok], v[ok])

auc_norm = {s: [sample_auc(PASS[s]['darts'], i) for i in range(8)] for s in ['chip z', 'chip p2-98']}
geo_chipz = [sample_auc(PASS['chip z']['geo'], i) for i in range(8)]
auc_norm = pd.DataFrame(auc_norm, index=SHORT)
auc_norm.insert(0, 'GeoAI raw', auc.GeoAI.values)
auc_norm['GeoAI chip z'] = geo_chipz
print('RTS AUC after per-chip normalisation (DARTS columns unless stated)'); print(auc_norm.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(18, 5))
h = ax[0].imshow(W.values.astype(float), cmap='magma_r', vmin=0, vmax=min(2, np.nanmax(W.values)))
ax[0].set_xticks(range(8), SHORT, rotation=45); ax[0].set_yticks(range(len(SCHEMES)), SCHEMES); ax[0].grid(False)
for (r, c), v in np.ndenumerate(W.values.astype(float)): ax[0].text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=8, color='w' if v > 1 else 'k')
plt.colorbar(h, ax=ax[0], label='Wasserstein (GeoAI σ)'); ax[0].set_title('distribution gap after each normalisation')
piv = dom[dom.texture].pivot(index='scheme', columns='group', values='domain AUC').loc[SCHEMES]
piv.plot.bar(ax=ax[1]); ax[1].axhline(.5, c='k', ls='--'); ax[1].axhline(.65, c='grey', ls=':')
ax[1].set_ylim(.4, 1.02); ax[1].set_ylabel('domain-classifier AUC (with texture)'); ax[1].set_title('can a classifier tell the source apart?')
plt.tight_layout(); save(fig, '08_normalisation_gap.png'); plt.show()

best_scheme = W[[SHORT[i] for i in OPTICAL]].mean(1).idxmin()
fig, ax = plt.subplots(2, 4, figsize=(19, 7.5))
for i, b in enumerate(SHORT):
    a_ = ax.flat[i]
    g, d = PASS[best_scheme]['geo']['pix'][:, i], PASS[best_scheme]['darts']['pix'][:, i]
    g, d = g[np.isfinite(g)], d[np.isfinite(d)]
    lo, hi = np.percentile(np.r_[g, d], [.5, 99.5]); bins = np.linspace(lo, hi, 80)
    a_.hist(g, bins, density=True, alpha=.55, color=C_GEO, label='GeoAI'); a_.hist(d, bins, density=True, alpha=.55, color=C_DARTS, label='DARTS')
    a_.set_title(f'{b} — {best_scheme}'); a_.legend(fontsize=8)
plt.tight_layout(); save(fig, '09_best_scheme_histograms.png'); plt.show()
print('best scheme for optical channels (lowest mean Wasserstein):', best_scheme)

## 7. Side-by-side gallery (chip p2–98 stretch, RTS outlined)

In [ ]:
def show(a_, im, m, kind):
    if kind == 'rgb':
        x = normalise(im, 'chip p2-98', 'x')[..., :3]; a_.imshow(np.nan_to_num(x))
    else:
        a_.imshow(im[..., kind], cmap='gray', vmin=np.nanpercentile(im[..., kind], 2), vmax=np.nanpercentile(im[..., kind], 98))
    a_.contour(m > 0, [.5], colors='r', linewidths=.7); a_.axis('off')

gi = rng.choice(len(geo_img), 6, replace=False); di = rng.choice(len(d_img), 6, replace=False)
fig, ax = plt.subplots(6, 6, figsize=(20, 17))
for r in range(6):
    for c, kind in enumerate(['rgb', 4, 5]):
        show(ax[r, c], geo_img[gi[r]], geo_inst[gi[r]], kind)
        show(ax[r, c + 3], d_img[di[r]], d_inst[di[r]], kind)
for c, t in enumerate(['GeoAI RGB', 'GeoAI rel elev', 'GeoAI shaded', 'DARTS RGB (S2)', 'DARTS rel elev', 'DARTS shaded (ESR)']):
    ax[0, c].set_title(t)
plt.tight_layout(); save(fig, '10_gallery.png'); plt.show()

## 8. Verdict

In [ ]:
best_by_group = dom[dom.texture].groupby('group').apply(lambda d: d.loc[d['domain AUC'].idxmin(), ['scheme', 'domain AUC']])
no_tex = dom[~dom.texture].groupby('group')['domain AUC'].min()
best_by_group['AUC without texture'] = no_tex
best_by_group['RTS cue agrees'] = [auc.loc[[SHORT[i] for i in GROUPS[g]], 'agree'].all() for g in best_by_group.index]
best_by_group['compatible (<0.65 & cue agrees)'] = (best_by_group['domain AUC'] < .65) & best_by_group['RTS cue agrees']
print(best_by_group.to_string())

_Interpretation is written below after the run (see the markdown cell that follows)._